<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiazlization

In [ ]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 MB 3.8 MB/s eta 0:00:00


In [ ]:
#DRIVE
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from llama_cpp import Llama, llama_free, llama_free_model
from tqdm import tqdm
#from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json
import torch

# Load the model
VERBOSE = True
CHOSEN = 'llama'
models = {
    'llama': {'repo_id':"bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
              'filename':"Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"},
    'qwen': {'repo_id':"Qwen/Qwen3-8B-GGUF",
             'filename': "Qwen3-8B-Q8_0.gguf"},
    'mistral': {'repo_id':"mistralai/Ministral-3-8B-Instruct-2512-GGUF",
                'filename':"Ministral-3-8B-Instruct-2512-Q8_0.gguf"},
}
file_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid_recharged']
iterations = 10
prompts = {}
model = Llama.from_pretrained(repo_id=models[CHOSEN]['repo_id'], # repository name
                            filename=models[CHOSEN]['filename'], # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)

def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

# Game parameter estimation
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration

## Estimating everything at once

In [ ]:
prompts['all'] = """You are a board‑game analyst that always explains its reasoning before answering.
For any supplied rule excerpt you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting already existing BGG mechanic(s).
3. Judge the rule density and decision depth. Then assign a complexity score (1‑5).
4. From the number of components in the box and player‑interaction patterns infer the optimal player‑count.
6. Assess the progression of a typical game turn. Based on the complexity of the required actions and how much each turn brings the player closer to the final goal, estimate the game’s average duration in minutes.
Explain your reasoning step by step then output a json object with the fields 'mechanics' (list of strings), 'complexity'(1-5), 'optimal player count', 'duration' that matches the schema below.

JSON schema:
{
  "type": "object",
  "properties": {
    "reasoning": {"type": "string"},
    "answer": {"type": "object", "properties": {
      "mechanics": {"type": "array", "items": {"type": "string"}},
      "complexity": {"type": "number", "minimum": 1, "maximum": 5},
      "optimal player count": {"type": "number"},
      "duration": {"type": "number"},
      "required": ["mechanics", "complexity", "optimal player count", "duration"]
    }}
  },
  "required": ["reasoning", "answer"]
}
"""



  2%|▎         | 1/40 [00:21<13:45, 21.16s/it]

**Observations**

Key actions:

1. Players draw train cards or tickets
2. Players claim routes on the board
3. Players discard cards or tickets
4. Players keep tickets and score points
5. Players calculate final scores

Components:

1. Game board (map of North American train routes)
2. Plastic trains in 5 colors
3. Scoring markers
4. Train cards
5. Locomotive cards (wild cards)
6. Tickets
7. Longest path bonus card

**Mechanics**

Based on the observations, the game features:

1. **Area Control**: Players claim routes on the board, which is a classic example of area control.
2. **Deck-Building**: Players draw train cards and locomotive cards, which can be used to claim routes.
3. **Ticket Collection**: Players collect and keep tickets, which are used to score points.
4. **Route-Building**: Players build continuous paths of plastic trains to claim routes and score points.

**Complexity**

The rule density is moderate, with a clear structure and rules for each action. The decision depth 

  5%|▌         | 2/40 [00:36<11:16, 17.80s/it]

**Key Actions and Components:**

1. Players take turns performing one of three actions: Draw Train Cards, Claim 1 Route, or Draw Tickets.
2. Players draw train cards from the deck or the top 5 face-up cards.
3. Players claim routes by playing train cards that match the color of the route.
4. Players draw tickets from the deck and must keep at least one.
5. Players score points for claiming routes, completing tickets, and having the longest continuous path of trains.
6. The game ends when a player's train stock is depleted, and players calculate their final scores.

**Mapped to BGG Mechanics:**

1. Area Control (Claiming routes and building a network of trains)
2. Card Drafting (Drawing train cards and tickets)
3. Route Building (Building a continuous path of trains)
4. Ticket Collection/Completion (Completing tickets to score points)

**Rule Density and Decision Depth:**

The rulebook is relatively dense, with a lot of information to absorb, but the game mechanics are not overly comple

  8%|▊         | 3/40 [00:49<09:43, 15.76s/it]

...

**Key Actions and Components:**

1. **Player Order:** Players determine their order based on the number of cities in their network or the size of their power plant.
2. **Power Plant Auction:** Players bid on power plants, with the lowest numbered plant being the cheapest.
3. **Resource Buying:** Players buy resources to power their plants.
4. **City Building:** Players expand their networks by connecting new cities.
5. **Bureaucracy:** Players produce electricity to supply their networks, earning cash.
6. **Resource Resupply:** Players resupply resources based on the number of players and the current Step.
7. **Power Plant Market:** Players draw new power plants from the stack, with the highest numbered plant being the most expensive.

**Mapped to BGG Mechanics:**

1. **Area Control:** Players compete to build their network by connecting cities.
2. **Resource Management:** Players manage resources to power their plants.
3. **Auction/Bidding:** Players bid on power plants to acquire the

 98%|█████████▊| 39/40 [11:53<00:19, 19.14s/it]

**Key Actions and Components:**

1. Players choose a contiguous playing zone of connected areas.
2. Players place houses on the board to connect cities.
3. Players buy power plants through auctions.
4. Players buy resources to power their power plants.
5. Players build houses to expand their electricity network.
6. Players produce electricity to supply their networks.
7. Players resupply the resource market.
8. Players update the power plant market.

**Mapped to BGG Mechanics:**

1. Modular board (choosing a contiguous playing zone).
2. Area control (placing houses on the board to connect cities).
3. Auction (buying power plants).
4. Resource management (buying resources to power power plants).
5. Network building (building houses to expand electricity network).
6. Engine building (producing electricity to supply networks).
7. Market manipulation (resupplying the resource market).
8. Card drafting (updating the power plant market).

**Rule Density and Decision Depth:**

The rule densit

100%|██████████| 40/40 [12:11<00:00, 18.29s/it]

**Key actions and components:**

* Players take turns in a specific order, determined by their number of cities and power plants
* Players buy power plants through auctions, with a discount token reducing the minimum bid for the smallest power plant
* Players buy resources from the resource market, with prices increasing over time
* Players build houses in their network, connecting new cities and expanding their network
* Players produce electricity to supply their network, earning cash based on the number of cities they power
* The game has three Steps, with different rules and restrictions for each Step

**BGG mechanics:**

* Area control (players control cities and expand their network)
* Auction (players bid on power plants)
* Resource management (players manage resources to produce electricity and build houses)
* Worker placement (players place houses in cities to build their network)

**Complexity score: 4**

The game has a moderate to high level of complexity due to the multiple

## Estimating each parameter separately

### Mechanics

In [ ]:
prompt = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. List the key actions and components you notice.
2. Map those observations to the most fitting BGG mechanic(s).

here is a complete list of BGG mechanics:
Acting, Action / Event, Action Drafting, Action Points, Action Queue, Action Retrieval,
Action Timer, Advantage Token, Alliances, Area Majority / Influence, Area Movement, Area-Impulse,
Auction / Bidding, Auction Compensation, Auction: Dexterity, Auction: Dutch, Auction: Dutch Priority,
Auction: English, Auction: Fixed Placement, Auction: Multiple Lot, Auction: Once Around,
Auction: Sealed Bid,Auction: Turn Order Until Pass, Automatic Resource Growth, Betting and Bluffing,
Bias, Bids As Wagers, Bingo, Bribery, Campaign / Battle Card Driven, Card Play Conflict Resolution,
Catch the Leader, Chaining, Chit-Pull System, Closed Drafting, Closed Economy Auction, Command Cards,
Commodity Speculation, Communication Limits, Connections, Constrained Bidding, Contracts,
Cooperative Game, Crayon Rail System, Critical Hits and Failures, Cube Tower, Deck Construction, "Deck, Bag, and Pool Building", Deduction,Delayed Purchase, Dice Rolling, Die Icon Resolution, Different Dice Movement, Drawing, Elapsed Real Time Ending, Enclosure, End Game Bonuses, Events, Finale Ending, Flicking, Follow, Force Commitment, Grid Coverage, Grid Movement, Hand Management, Hexagon Grid, Hidden Movement, Hidden Roles, Hidden Victory Points, Highest-Lowest Scoring, Hot Potato, "I Cut, You Choose", Impulse Movement, Income, Increase Value of Unchosen Resources, Induction, Interrupts, Investment, Kill Steal, King of the Hill, Ladder Climbing, Layering, Legacy Game, Line Drawing, Line of Sight, Loans, Lose a Turn, Mancala,
Map Addition, Map Deformation, Map Reduction, Market, Matching, Measurement Movement,
Melding and Splaying, Memory, Minimap Resolution, Modular Board, Move Through Deck,
Movement Points, Movement Template, Moving Multiple Units, Multi-Use Cards, Multiple Maps,
Narrative Choice / Paragraph, Negotiation, Neighbor Scope, Network and Route Building,
Once-Per-Game Abilities, Open Drafting, Order Counters, Ordering, Ownership, Paper-and-Pencil,
Passed Action Token, Pattern Building, Pattern Movement, Pattern Recognition, Physical Removal,
Pick-up and Deliver, Pieces as Map, Player Elimination, Player Judge, Point to Point Movement,
Predictive Bid, Prisoner's Dilemma, Programmed Movement, Push Your Luck, Questions and Answers, Race,
Random Production, Ratio / Combat Results Table, Re-rolling and Locking, Real-Time, Relative Movement,
Resource Queue, Resource to Move, Rock-Paper-Scissors, Role Playing, Roles with Asymmetric Information,
Roll / Spin and Move, Rondel, Scenario / Mission / Campaign Game, Score-and-Reset Game, Secret Unit Deployment,
Selection Order Bid, Semi-Cooperative Game, Set Collection, Simulation, Simultaneous Action Selection,
Singing, Single Loser Game, Slide / Push, Solo / Solitaire Game, Speed Matching, Spelling, Square Grid,
Stacking and Balancing, Stat Check Resolution, Static Capture, Stock Holding, Storytelling, Sudden Death Ending,
Tags, Take That, Targeted Clues, Team-Based Game, Tech Trees / Tech Tracks, Three Dimensional Movement,
Tile Placement, Track Movement, Trading, Traitor Game, Trick-taking, Tug of War, Turn Order: Auction,
Turn Order: Claim Action, Turn Order: Pass Order, Turn Order: Progressive, Turn Order: Random,
Turn Order: Role Order, Turn Order: Stat-Based, Turn Order: Time Track, Variable Phase Order,
Variable Player Powers, Variable Set-up, Victory Points as a Resource, Voting, Worker Placement,
Worker Placement with Dice Workers, "Worker Placement, Different Worker Types", Zone of Control
"""


  2%|▎         | 1/40 [00:20<13:12, 20.32s/it]

**Key Actions and Components:**

1. Drawing train cards (face-up or from the deck)
2. Claiming a route (playing train cards to link cities)
3. Drawing tickets (revealing new ticket objectives)
4. Discarding train cards to claim a route
5. Discarding tickets (keeping a minimum of 1)
6. Revealing tickets (showing completed or incomplete tickets)
7. Scoring points (routes, tickets, and longest path bonus)

**Components:**

1. Train cards (110 total, 8 colors + 14 locomotive cards)
2. Plastic trains (5 colors, 45 per player)
3. Scoring markers (5, one per player)
4. Game board (map of North American train routes)
5. Longest path bonus card
6. Ticket deck (33 tickets)
7. Train deck (110 train cards)

**Notable Mechanics:**

1. **Variable Phase Order**: Players can choose to perform one of three actions on their turn (draw train cards, claim a route, or draw tickets).
2. **Resource Queue**: Train cards are drawn and discarded, but the player can choose to play them to claim a route.
3. **Rou

  5%|▌         | 2/40 [00:41<13:01, 20.56s/it]

**Key Actions and Components:**

1. Drawing train cards (from deck or face-up cards)
2. Claiming routes (by playing train cards)
3. Drawing tickets (by drawing 3 from the ticket deck)
4. Keeping and discarding tickets
5. Revealing completed tickets at the end of the game
6. Scoring points for routes and tickets
7. Determining the longest continuous path of trains for bonus points

**Mapped to BGG Mechanics:**

1. Drawing train cards:
	* Mechanic: Draw Train Cards (from deck or face-up cards)
	* This action is repeated throughout the game, with the possibility of drawing from the face-up cards or the deck.
2. Claiming routes:
	* Mechanic: Tile Placement (by playing train cards to create a route)
	* This mechanic is used to create a path of connected spaces on the board.
3. Drawing tickets:
	* Mechanic: Drawing (by drawing 3 tickets from the ticket deck)
	* This action allows players to acquire new tickets to complete.
4. Keeping and discarding tickets:
	* Mechanic: Hand Management (by s

  8%|▊         | 3/40 [00:57<11:27, 18.58s/it]

...

Key actions and components:

* Players determine their playing zone and choose their initial city.
* Players draw a number of resource tokens and place them in the resource market.
* Players take turns buying power plants, resources, and building houses to expand their network.
* Players earn money by supplying electricity to their cities and resupply the resource market.
* The game has three steps, each with its own rules and resource resupply.
* Players can connect multiple cities to their network, with increasing costs as the step progresses.
* Players can store resources on their power plants and move them between plants.
* The game ends when a player connects a certain number of cities, and the player who can power the most cities wins.

Mechanics:

* **Worker Placement**: Players place their houses on the board to represent their cities and expand their network.
* **Resource Management**: Players manage resources (coal, oil, garbage, and uranium) to produce electricity and supply

 98%|█████████▊| 39/40 [14:25<00:24, 24.07s/it]

**Key Actions and Components:**

* Players take turns buying power plants, resources, and building cities.
* Power plants are auctioned off, with players bidding to buy them.
* Resources are purchased from the market, with prices determined by the number of players.
* Players build cities by connecting new ones to their network, with costs determined by the number of players.
* Players earn cash by producing electricity for their cities.
* The game has three steps, with different rules for each step.

**BGG Mechanics:**

Based on the analysis of the game mechanics, the following BGG mechanics are identified:

1. **Action Point Allowance System**: Players take turns buying power plants, resources, and building cities, with each turn representing a specific action.
2. **Auction**: Power plants are auctioned off, with players bidding to buy them.
3. **Resource Management**: Players purchase resources from the market, with prices determined by the number of players.
4. **Area Control**: Pl

100%|██████████| 40/40 [14:51<00:00, 22.29s/it]

**Key actions and components:**

1. Players take turns in a specific order, determined by the number of cities in their network (Phase 1: Determine Player Order).
2. Players may buy power plants through auctions, with the first player choosing between offering a power plant or passing (Phase 2: Auction Power Plants).
3. Players buy resources for their power plants from the resource market, with prices determined by the number of players and the current Step of the game (Phase 3: Buy Resources).
4. Players build houses to expand their electricity networks, with costs determined by the number of players and the current Step of the game (Phase 4: Build Houses).
5. Players produce electricity to supply their networks and earn cash, with resources resupplied according to the current Step of the game (Phase 5: Bureaucracy).

**Components:**

1. Power plants with different resource requirements and capacities.
2. Resource tokens (coal, oil, garbage, uranium).
3. Houses to build electricity ne

### Complexity rating

In [ ]:
prompt = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Analyze Learning Complexity
- Analyze the length of the text, setup steps, rule exceptions and other factor that may indicate how rule intensive the game is.
- Reason about how quickly a new player could grasp the basics.
2. Analyze Playing Complexity
- Look at in‑game actions per turn, resource management, simultaneous moves and number of element to manage
- Estimate mental load during a typical play session.
3. Analyze Strategy/Tactics
- Examine depth of decision space, long‑term planning, branching possibilities, and how impactful is a wrong decision.
4. Convert each qualitative assessment to a numeric rating (1‑5)
- Provide a short justification for each number.
5 Compute the complexity rating
- Average = (Learning + Playing + Strategy) / 3
- Round to one decimal.

**Final Output**
Learning: X
Playing: Y
Strategy: Z
Overall Complexity: W.W"""

### Optimal player count

In [ ]:
prompt = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Identify the player‑count range stated in the rules (minimum‑maximum). If the rulebook does not explicitly state a player count, infer the appropriate range from the game components and mechanics described in the box contents.
2. Examine how the core mechanics scale with player number
3. Consider the impact on play time, player interaction, and variance (e.g., games that become chaotic with many players or too slow with few).
4. Weigh the pros and cons of each possible player count within the allowed range.
5. find the optimal player count based on your observations

**Final Output**
Optimal player count: X
"""

### Game duration

In [ ]:
prompt = """You are a board game analyst that always explains its reasoning before answering.
For any supplied rulebook you must:
1. Assess the progression of a typical game turn.
2. Evaluate the complexity of the required actions and how long a turn would last
3. Evaluate how much each turn brings the player closer to the final goal
4. Based on your observation estimate the game’s average duration in minutes.

**Final Output**
Average game duration: X minutes
"""

# Execution

In [ ]:
res = []
output_dict = {g: {it: '' for it in range(iterations)} for g in file_names}
for game,it in tqdm([(f,it) for f in file_names for it in range(iterations)]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
    out = model.create_chat_completion(generate_message(prompt, rulebook),
                                       temperature=0.7,
                                       response_format = {"type": "json_object"},
                                       stop=["\n\n"] # to avoid other text after the json
                                       )['choices'][0]['message']['content']
    output_dict[game][it] = out
    if(VERBOSE):
        print(out)
    res.append(out.splitlines()[-1].split(','))

with open(f'drive/MyDrive/NLP_proj/{CHOSEN}_estimation.out','w') as f:
    json.dump(dict(output_dict),f)

In [ ]:
res = []
output_dict = {g: {it: '' for it in range(iterations)} for g in file_names}

for game,it in tqdm([(f,it) for f in file_names for it in range(iterations)]):
    rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
    out = model.create_chat_completion(generate_message(prompt, rulebook), temperature=0.7)['choices'][0]['message']['content']
    output_dict[game][it] = out
    if(VERBOSE):
        print(out)
    res.append(out.splitlines()[-1].split(','))
print(res)
with open(f'drive/MyDrive/NLP_proj/{CHOSEN}_estimation_mechanics.out','w') as f:
    json.dump(dict(output_dict),f)